In [ ]:
### Resource Credentials

In [1]:
%pip install azure-ai-vision-imageanalysis azure-core matplotlib


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from azure.ai.vision.imageanalysis import ImageAnalysisClient
from azure.ai.vision.imageanalysis.models import VisualFeatures
from azure.core.credentials import AzureKeyCredential

In [ ]:
# Clean the strings
key = "<REDACTED_API_KEY>"
endpoint = "https://temp-east-azure-ai.cognitiveservices.azure.com/"

In [4]:
# 1. FORCE CLEAN THE STRINGS (Ensures no hidden spaces or tabs)
clean_key = key.strip()
clean_endpoint = endpoint.strip().rstrip("/")
print(f"Connecting to: {clean_endpoint}...")

client = ImageAnalysisClient(
    endpoint=clean_endpoint, 
    credential=AzureKeyCredential(clean_key)
)

Connecting to: https://temp-east-azure-ai.cognitiveservices.azure.com...


### Unified Analysis (Multi-Feature Request)

In [5]:
import os

# Use a Rush album cover for the test
image_path = "rush_album_covers/Signals.png"

with open(image_path, "rb") as f:
    image_data = f.read()

# Requesting multiple features
result = client.analyze(
    image_data=image_data,
    visual_features=[
        VisualFeatures.READ,
        VisualFeatures.TAGS,
        VisualFeatures.PEOPLE,
        VisualFeatures.SMART_CROPS
    ]
)

# 1. Process OCR (Read)
if result.read is not None:
    print("--- OCR Result ---")
    for block in result.read.blocks:
        for line in block.lines:
            print(f"Detected Text: '{line.text}'")

# 2. Process Tags
if result.tags is not None:
    print("\n--- Top 5 Tags ---")
    for tag in result.tags.list[:5]:
        print(f"Tag: {tag.name} (Confidence: {tag.confidence:.2f})")

# 3. Process People
if result.people is not None:
    print(f"\n--- People Detected: {len(result.people.list)} ---")
    for person in result.people.list:
        print(f"Person detected with confidence: {person.confidence:.2f}")

# 4. Process Smart Crop
if result.smart_crops is not None:
    print("\n--- Smart Crop (Area of Interest) ---")
    for crop in result.smart_crops.list:
        print(f"Aspect Ratio {crop.aspect_ratio}: Bounding Box {crop.bounding_box}")

--- OCR Result ---
Detected Text: '٦'
Detected Text: 'G'
Detected Text: 'N'
Detected Text: 'S'
Detected Text: 'S'

--- Top 5 Tags ---
Tag: animal (Confidence: 0.97)
Tag: dog (Confidence: 0.92)
Tag: grass (Confidence: 0.91)

--- People Detected: 0 ---

--- Smart Crop (Area of Interest) ---
Aspect Ratio 1.55: Bounding Box {'x': 22, 'y': 110, 'w': 480, 'h': 310}


### Advanced OCR: Synchronous Multi-Language Read

In [6]:
# Synchronous OCR call
read_result = client.analyze(
    image_data=image_data,
    visual_features=[VisualFeatures.READ]
)

if read_result.read:
    full_text = " ".join([line.text for block in read_result.read.blocks for line in block.lines])
    print(f"Extracted Text: {full_text}")

Extracted Text: ٦ G N S S


### Multimodal Embeddings (Vectorization)

In [7]:
# Note: Ensure your SDK version supports the 'IMAGE_ANALYSIS_EMBEDDINGS' preview or GA feature
# In GA v4.0, this is often handled via specialized calls or the retrieval API

from azure.ai.vision.imageanalysis.models import VisualFeatures

# Generate Embedding
embed_result = client.analyze(
    image_data=image_data,
    visual_features=[VisualFeatures.TAGS] # Standard tags are float-based
)
